# C／C++ 共同基礎進階 100 題：自動檢查版

這份 Notebook 改成和 **Python Loops Functions Lists Questions.ipynb** 相同的作答流程：

1. 先執行「自動檢查工具」。
2. 每題只修改 C 或 C++ 的作答 cell。
3. **不用自己在執行時輸入測試資料。**
4. 完成後執行該語言下方的「檢查」cell。
5. Notebook 會自動把題目的測試輸入送進程式，並比較輸出。
6. 最後執行「總檢查」查看通過題數。

## 作答方式

每題提供兩種語言，你可以選擇其中一種：

- `%%c_answer Q1`：登記這題的 C 程式
- `%%cpp_answer Q1`：登記這題的 C++ 程式

和 Python 題本一樣，**Answer cell 本身不要求你手動輸入 stdin**；輸入由檢查 cell 自動提供。

> 本版本保留原始 100 題的題目、輸入輸出規格與範例。檢查工具目前使用各題題目中提供的範例測試資料。


In [ ]:
# === C / C++ 自動檢查工具：請先執行這個 cell，不要修改 ===
import base64
import json
import os
import shutil
import subprocess
import tempfile
from IPython.core.magic import register_cell_magic

_TEST_CASES = json.loads(
    base64.b64decode("eyJRMSI6IFt7ImlucHV0IjogIjEwIDIwIDMwXG4iLCAiZXhwZWN0ZWQiOiAiMjAgMzAgMTBcbiJ9XSwgIlEyIjogW3siaW5wdXQiOiAiNyA0IDEyXG4iLCAiZXhwZWN0ZWQiOiAiNTFcbiJ9XSwgIlEzIjogW3siaW5wdXQiOiAiMyA1IDIwXG4iLCAiZXhwZWN0ZWQiOiAiMTAxMFxuIn1dLCAiUTQiOiBbeyJpbnB1dCI6ICIyIC0zIDQgNSA2XG4iLCAiZXhwZWN0ZWQiOiAiMzUzXG4ifV0sICJRNSI6IFt7ImlucHV0IjogIjgwIDc1IDkyIDg4IDk1XG4xIDEgMiAyIDRcbiIsICJleHBlY3RlZCI6ICI4OS41MFxuIn1dLCAiUTYiOiBbeyJpbnB1dCI6ICIxMiAxOFxuIiwgImV4cGVjdGVkIjogIjE0LjQwMDBcbiJ9XSwgIlE3IjogW3siaW5wdXQiOiAiMiA4IDMyXG4iLCAiZXhwZWN0ZWQiOiAiOC4wMDBcbiJ9XSwgIlE4IjogW3siaW5wdXQiOiAiMSAyIDExIDE3IDIgM1xuIiwgImV4cGVjdGVkIjogIjUuMDAgOC4wMFxuIn1dLCAiUTkiOiBbeyJpbnB1dCI6ICIzIC0xMCA0XG4iLCAiZXhwZWN0ZWQiOiAiNTJcbiJ9XSwgIlExMCI6IFt7ImlucHV0IjogIjEgLTUgNlxuIiwgImV4cGVjdGVkIjogIjMuMDAwIDIuMDAwXG4ifV0sICJRMTEiOiBbeyJpbnB1dCI6ICI0NzBcbiIsICJleHBlY3RlZCI6ICI3NFxuIn1dLCAiUTEyIjogW3siaW5wdXQiOiAiMjMwN1xuIiwgImV4cGVjdGVkIjogIjEyIDBcbiJ9XSwgIlExMyI6IFt7ImlucHV0IjogIjkgMCAyIDYgMSA0XG4iLCAiZXhwZWN0ZWQiOiAiOTAyNjE0XG4ifV0sICJRMTQiOiBbeyJpbnB1dCI6ICI1ODMxXG4iLCAiZXhwZWN0ZWQiOiAiNTM4MVxuIn1dLCAiUTE1IjogW3siaW5wdXQiOiAiMzE0MjdcbiIsICJleHBlY3RlZCI6ICI3MTQyM1xuIn1dLCAiUTE2IjogW3siaW5wdXQiOiAiMTAwMjVcbiIsICJleHBlY3RlZCI6ICIyOjQ3OjVcbiJ9XSwgIlExNyI6IFt7ImlucHV0IjogIjM3MjM0NTZcbiIsICJleHBlY3RlZCI6ICIxIDIgMyA0NTZcbiJ9XSwgIlExOCI6IFt7ImlucHV0IjogIjM1MDBcbiIsICJleHBlY3RlZCI6ICIyIDEwIDIwXG4ifV0sICJRMTkiOiBbeyJpbnB1dCI6ICIyMyA1MCAxMzVcbiIsICJleHBlY3RlZCI6ICIyIDVcbiJ9XSwgIlEyMCI6IFt7ImlucHV0IjogIjUgMTAwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTIxIjogW3siaW5wdXQiOiAiMTM3IDEyXG4iLCAiZXhwZWN0ZWQiOiAiMTEgNVxuIn1dLCAiUTIyIjogW3siaW5wdXQiOiAiMjg3XG4iLCAiZXhwZWN0ZWQiOiAiNSAzIDEgMlxuIn1dLCAiUTIzIjogW3siaW5wdXQiOiAiMzY4NlxuIiwgImV4cGVjdGVkIjogIjMgMSAxIDEgMyAxIDFcbiJ9XSwgIlEyNCI6IFt7ImlucHV0IjogIjEyMzQgMTIwXG4iLCAiZXhwZWN0ZWQiOiAiMTAgMzRcbiJ9XSwgIlEyNSI6IFt7ImlucHV0IjogIjEwNDg1NzkgNDA5NlxuIiwgImV4cGVjdGVkIjogIjI1NiAzXG4ifV0sICJRMjYiOiBbeyJpbnB1dCI6ICIxMiAxMCAxN1xuIiwgImV4cGVjdGVkIjogIjNcbiJ9XSwgIlEyNyI6IFt7ImlucHV0IjogIjEwIDMgN1xuIiwgImV4cGVjdGVkIjogIjZcbiJ9XSwgIlEyOCI6IFt7ImlucHV0IjogIjdcbiIsICJleHBlY3RlZCI6ICIyIDFcbiJ9XSwgIlEyOSI6IFt7ImlucHV0IjogIjcgMTEgMjBcbiIsICJleHBlY3RlZCI6ICIxNTFcbiJ9XSwgIlEzMCI6IFt7ImlucHV0IjogIjEyIDIwMCA0NVxuIiwgImV4cGVjdGVkIjogIjgzNzY3N1xuIn1dLCAiUTMxIjogW3siaW5wdXQiOiAiODM3Njc3XG4iLCAiZXhwZWN0ZWQiOiAiMTIgMjAwIDQ1XG4ifV0sICJRMzIiOiBbeyJpbnB1dCI6ICI3IDEyXG4iLCAiZXhwZWN0ZWQiOiAiMC41ODMzM1xuIn1dLCAiUTMzIjogW3siaW5wdXQiOiAiMTAgMTEgMTIgMTMgMTdcbiIsICJleHBlY3RlZCI6ICIxMi42MFxuIn1dLCAiUTM0IjogW3siaW5wdXQiOiAiMTIzLjQ1NjdcbiIsICJleHBlY3RlZCI6ICIxMjMgMC40NTY3XG4ifV0sICJRMzUiOiBbeyJpbnB1dCI6ICI3OC42NVxuIiwgImV4cGVjdGVkIjogIjc5XG4ifV0sICJRMzYiOiBbeyJpbnB1dCI6ICIzNyA0NVxuIiwgImV4cGVjdGVkIjogIjgyLjIyJVxuIn1dLCAiUTM3IjogW3siaW5wdXQiOiAiMzYuNVxuIiwgImV4cGVjdGVkIjogIjk3LjcwIDMwOS42NVxuIn1dLCAiUTM4IjogW3siaW5wdXQiOiAiMTIzLjQ1NlxuIiwgImV4cGVjdGVkIjogIjEyMyA0NlxuIn1dLCAiUTM5IjogW3siaW5wdXQiOiAiNy40MjVcbiIsICJleHBlY3RlZCI6ICI3IDI2XG4ifV0sICJRNDAiOiBbeyJpbnB1dCI6ICI2MTIuNSA0Mi44XG4iLCAiZXhwZWN0ZWQiOiAiMTQuMzExXG4ifV0sICJRNDEiOiBbeyJpbnB1dCI6ICI2MTIuNSA0Mi44XG4iLCAiZXhwZWN0ZWQiOiAiNi45OVxuIn1dLCAiUTQyIjogW3siaW5wdXQiOiAiOTg3NjU0MzIxMCA3My41XG4iLCAiZXhwZWN0ZWQiOiAiMTI4LjE0OTdcbiJ9XSwgIlE0MyI6IFt7ImlucHV0IjogIjEwLjUgNDIuMCAwLjM1XG4iLCAiZXhwZWN0ZWQiOiAiMjEuNTI1XG4ifV0sICJRNDQiOiBbeyJpbnB1dCI6ICJxXG4iLCAiZXhwZWN0ZWQiOiAiUVxuIn1dLCAiUTQ1IjogW3siaW5wdXQiOiAiTVxuIiwgImV4cGVjdGVkIjogIm1cbiJ9XSwgIlE0NiI6IFt7ImlucHV0IjogIlRcbiIsICJleHBlY3RlZCI6ICIyMFxuIn1dLCAiUTQ3IjogW3siaW5wdXQiOiAiMjJcbiIsICJleHBlY3RlZCI6ICJWXG4ifV0sICJRNDgiOiBbeyJpbnB1dCI6ICJYIDMxXG4iLCAiZXhwZWN0ZWQiOiAiQ1xuIn1dLCAiUTQ5IjogW3siaW5wdXQiOiAiN1xuIiwgImV4cGVjdGVkIjogIjcgNDlcbiJ9XSwgIlE1MCI6IFt7ImlucHV0IjogIjVcbiIsICJleHBlY3RlZCI6ICI1IDZcbiJ9XSwgIlE1MSI6IFt7ImlucHV0IjogIkYgUlxuIiwgImV4cGVjdGVkIjogIjEyXG4ifV0sICJRNTIiOiBbeyJpbnB1dCI6ICJEIEtcbiIsICJleHBlY3RlZCI6ICI4OFxuIn1dLCAiUTUzIjogW3siaW5wdXQiOiAiMzQ3XG4iLCAiZXhwZWN0ZWQiOiAiTkpcbiJ9XSwgIlE1NCI6IFt7ImlucHV0IjogIjguNCA1LjdcbiIsICJleHBlY3RlZCI6ICIyOC4yMCA0Ny44OCAxMC4xNVxuIn1dLCAiUTU1IjogW3siaW5wdXQiOiAiNy41IDguMiA5LjFcbiIsICJleHBlY3RlZCI6ICIyOS4wMjBcbiJ9XSwgIlE1NiI6IFt7ImlucHV0IjogIjEwIDYuNVxuIiwgImV4cGVjdGVkIjogIjE4MS40MjcwXG4ifV0sICJRNTciOiBbeyJpbnB1dCI6ICIzLjIgMTEuNVxuIiwgImV4cGVjdGVkIjogIjI5NS41NjEgMzY5Ljk1NFxuIn1dLCAiUTU4IjogW3siaW5wdXQiOiAiNS41IDEyXG4iLCAiZXhwZWN0ZWQiOiAiMTMuMjAwIDM4MC4xMzNcbiJ9XSwgIlE1OSI6IFt7ImlucHV0IjogIi0yLjUgNCA4LjUgLTNcbiIsICJleHBlY3RlZCI6ICIxMy4wMzggMy4wMDAgMC41MDBcbiJ9XSwgIlE2MCI6IFt7ImlucHV0IjogIjEgMiAzIDcgLTIgMTFcbiIsICJleHBlY3RlZCI6ICIxMC43NzAzXG4ifV0sICJRNjEiOiBbeyJpbnB1dCI6ICIyIDUgOCAyM1xuIiwgImV4cGVjdGVkIjogIjMuMDAwIC0xLjAwMFxuIn1dLCAiUTYyIjogW3siaW5wdXQiOiAiMyA0IC0yIDVcbiIsICJleHBlY3RlZCI6ICIxNC4wMDAgMjYuOTI2XG4ifV0sICJRNjMiOiBbeyJpbnB1dCI6ICIwIDAgOSAzIDMgMTJcbiIsICJleHBlY3RlZCI6ICI0LjAwMCA1LjAwMFxuIn1dLCAiUTY0IjogW3siaW5wdXQiOiAiNzIuNSAxNzYgMjJcbiIsICJleHBlY3RlZCI6ICIyMy40MSAtNC4zNVxuIn1dLCAiUTY1IjogW3siaW5wdXQiOiAiMTIuNSA3LjJcbiIsICJleHBlY3RlZCI6ICI5MC4wMDAgMzI0LjAwMFxuIn1dLCAiUTY2IjogW3siaW5wdXQiOiAiMi40IDE4LjVcbiIsICJleHBlY3RlZCI6ICI0MzUuNDE1IDE5LjA0OVxuIn1dLCAiUTY3IjogW3siaW5wdXQiOiAiNDUgMThcbiIsICJleHBlY3RlZCI6ICIzLjAyOSA1NC41MzBcbiJ9XSwgIlE2OCI6IFt7ImlucHV0IjogIjMwIDM4XG4iLCAiZXhwZWN0ZWQiOiAiODkuMDQ4XG4ifV0sICJRNjkiOiBbeyJpbnB1dCI6ICIxMDAgMjIwIDMzMFxuIiwgImV4cGVjdGVkIjogIjU2Ljg5NjZcbiJ9XSwgIlE3MCI6IFt7ImlucHV0IjogIjEyIDguMlxuIiwgImV4cGVjdGVkIjogIjEuNDYzIDE3LjU2MVxuIn1dLCAiUTcxIjogW3siaW5wdXQiOiAiMi41IDMxMCAxOFxuIiwgImV4cGVjdGVkIjogIjMuNTMzMFxuIn1dLCAiUTcyIjogW3siaW5wdXQiOiAiMC4wMDAwMVxuIiwgImV4cGVjdGVkIjogIjcwLjAwXG4ifV0sICJRNzMiOiBbeyJpbnB1dCI6ICIwLjAwMDAwMzJcbiIsICJleHBlY3RlZCI6ICI1LjQ5NVxuIn1dLCAiUTc0IjogW3siaW5wdXQiOiAiMi41IDgwIDQuMCAyMFxuIiwgImV4cGVjdGVkIjogIjQzLjA4XG4ifV0sICJRNzUiOiBbeyJpbnB1dCI6ICIxNTAwMDAgMi40IDMuNVxuIiwgImV4cGVjdGVkIjogIjEyNjAwLjAwIDE2MjYwMC4wMFxuIn1dLCAiUTc2IjogW3siaW5wdXQiOiAiODAwMDAgMy4yNSA2XG4iLCAiZXhwZWN0ZWQiOiAiOTY5MjMuNzhcbiJ9XSwgIlE3NyI6IFt7ImlucHV0IjogIjI0OTkgMTggNVxuIiwgImV4cGVjdGVkIjogIjIwNDkuMTggMjE1MS42NFxuIn1dLCAiUTc4IjogW3siaW5wdXQiOiAiMzg2MCAxMCA1IDdcbiIsICJleHBlY3RlZCI6ICI0NDU4LjMwIDYzNi45MFxuIn1dLCAiUTc5IjogW3siaW5wdXQiOiAiMjUwMDAgMC4wMzA4IDE0OC42XG4iLCAiZXhwZWN0ZWQiOiAiNzcwLjAwIDExNDQyMi4wMFxuIn1dLCAiUTgwIjogW3siaW5wdXQiOiAiNzIwMCAxODAgMTIuNSAyNFxuIiwgImV4cGVjdGVkIjogIjY0ODAuMDAgMjcwLjAwXG4ifV0sICJRODEiOiBbeyJpbnB1dCI6ICIzODAwMDAgNTAwMCAxLjc1XG4iLCAiZXhwZWN0ZWQiOiAiNjY1MC4wMCAxMTY1MC4wMFxuIn1dLCAiUTgyIjogW3siaW5wdXQiOiAiMzguNSA1OS45IDI0MFxuIiwgImV4cGVjdGVkIjogIjE0Mzc2LjAwIDkyNDAuMDAgNTEzNi4wMCA1NS41OFxuIn1dLCAiUTgzIjogW3siaW5wdXQiOiAiNyAtMiA1IDlcbiIsICJleHBlY3RlZCI6ICI3M1xuIn1dLCAiUTg0IjogW3siaW5wdXQiOiAiMSAyIDMgNFxuNSA2IDcgOFxuIiwgImV4cGVjdGVkIjogIjE5IDIyXG40MyA1MFxuIn1dLCAiUTg1IjogW3siaW5wdXQiOiAiMiAzIDEzIDUgLTIgNFxuIiwgImV4cGVjdGVkIjogIjIuMDAwIDMuMDAwXG4ifV0sICJRODYiOiBbeyJpbnB1dCI6ICIzLjUgLTIgNy4yNSA0LjVcbiIsICJleHBlY3RlZCI6ICIxMC43NSAyLjUwXG4tMy43NSAtNi41MFxuIn1dLCAiUTg3IjogW3siaW5wdXQiOiAiMyAtNCAyLjUgMS41XG4iLCAiZXhwZWN0ZWQiOiAiMTMuNTAgLTUuNTBcbiJ9XSwgIlE4OCI6IFt7ImlucHV0IjogIjE1NyAyNFxuIiwgImV4cGVjdGVkIjogIjYgMTMgMjRcbiJ9XSwgIlE4OSI6IFt7ImlucHV0IjogIjUgMTIgNyAxOFxuIiwgImV4cGVjdGVkIjogIjE3NCAyMTZcbiJ9XSwgIlE5MCI6IFt7ImlucHV0IjogIjIgMyA1IDcgOSAxMVxuIiwgImV4cGVjdGVkIjogIjkwIDIzMVxuIn1dLCAiUTkxIjogW3siaW5wdXQiOiAiMzAgMTgwIDI0MFxuIiwgImV4cGVjdGVkIjogIjE0MiAxNFxuIn1dLCAiUTkyIjogW3siaW5wdXQiOiAiNDgyNzMxXG4iLCAiZXhwZWN0ZWQiOiAiN1xuIn1dLCAiUTkzIjogW3siaW5wdXQiOiAiMTIzNDU2Nzg5XG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn1dLCAiUTk0IjogW3siaW5wdXQiOiAiOCA0NyAzNSAxNiAxMiA5XG4iLCAiZXhwZWN0ZWQiOiAiNyAyNCAzNFxuIn1dLCAiUTk1IjogW3siaW5wdXQiOiAiMjIgNTggNDcgMTAwMDBcbiIsICJleHBlY3RlZCI6ICIxIDQ1IDI3XG4ifV0sICJROTYiOiBbeyJpbnB1dCI6ICIzIC0yIDUgNCAxLjVcbiIsICJleHBlY3RlZCI6ICItMy4wMCAxMi4wMFxuIn1dLCAiUTk3IjogW3siaW5wdXQiOiAiMTkyMCAxMDgwIDYyLjVcbiIsICJleHBlY3RlZCI6ICIxMjAwIDY3NSA4MTAwMDBcbiJ9XSwgIlE5OCI6IFt7ImlucHV0IjogIjg1MCAxMjAgMzdcbiIsICJleHBlY3RlZCI6ICI1OS40NTZcbiJ9XSwgIlE5OSI6IFt7ImlucHV0IjogIjEyMCA4MCAxNTAgMiAxLjUgM1xuIiwgImV4cGVjdGVkIjogIjUzLjg0NiA1NC40NDRcbiJ9XSwgIlExMDAiOiBbeyJpbnB1dCI6ICIxMjkuOSAxOCAxMi41IDgwIDUgNFxuIiwgImV4cGVjdGVkIjogIjIzMzguMjAgMjA0NS45MyAyMjMyLjIyIDU1OC4wNlxuIn1dfQ==").decode("utf-8")
)

_C_ANSWERS = {}
_CPP_ANSWERS = {}
_RESULTS = {}

def _normalise_output(text):
    lines = [line.rstrip() for line in text.replace("\r\n", "\n").split("\n")]
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)

def _compiler(language):
    if language == "c":
        candidates = ["gcc", "clang", "cc"]
    else:
        candidates = ["g++", "clang++", "c++"]
    for name in candidates:
        path = shutil.which(name)
        if path:
            return path
    return None

def _compile_and_run(source, language, input_text):
    compiler = _compiler(language)
    if compiler is None:
        return False, "", "找不到 C/C++ 編譯器。"

    suffix = ".c" if language == "c" else ".cpp"
    standard = "-std=c11" if language == "c" else "-std=c++17"

    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, "answer" + suffix)
        exe = os.path.join(td, "answer")

        with open(src, "w", encoding="utf-8") as f:
            f.write(source)

        compile_result = subprocess.run(
            [compiler, standard, src, "-lm", "-o", exe],
            capture_output=True,
            text=True,
        )

        if compile_result.returncode != 0:
            return False, "", compile_result.stderr

        try:
            run_result = subprocess.run(
                [exe],
                input=input_text,
                capture_output=True,
                text=True,
                timeout=5,
            )
        except subprocess.TimeoutExpired:
            return False, "", "程式執行超過 5 秒。"

        if run_result.returncode != 0:
            err = run_result.stderr or f"程式結束碼：{run_result.returncode}"
            return False, run_result.stdout, err

        return True, run_result.stdout, ""

@register_cell_magic
def c_answer(line, cell):
    qid = line.strip().upper()
    _C_ANSWERS[qid] = cell
    print(f"已儲存 {qid} 的 C 作答。請執行下方檢查 cell。")

@register_cell_magic
def cpp_answer(line, cell):
    qid = line.strip().upper()
    _CPP_ANSWERS[qid] = cell
    print(f"已儲存 {qid} 的 C++ 作答。請執行下方檢查 cell。")

def _run_check(qid, language):
    qid = qid.upper()
    answers = _C_ANSWERS if language == "c" else _CPP_ANSWERS
    label = "C" if language == "c" else "C++"

    if qid not in answers:
        print(f"❌ {qid}：尚未執行 {label} 作答 cell")
        _RESULTS[(qid, language)] = False
        return

    source = answers[qid]
    cases = _TEST_CASES[qid]
    passed = 0

    for case_number, case in enumerate(cases, start=1):
        ok_run, stdout, error = _compile_and_run(
            source, language, case["input"]
        )

        if not ok_run:
            print(f"❌ {qid} {label}：編譯或執行失敗")
            print(error)
            _RESULTS[(qid, language)] = False
            return

        actual = _normalise_output(stdout)
        expected = _normalise_output(case["expected"])

        if actual == expected:
            passed += 1
        else:
            print(f"❌ {qid} {label} 測試 {case_number} 未通過")
            print("測試輸入：")
            print(case["input"], end="")
            print("你的輸出：")
            print(stdout if stdout else "<沒有輸出>")
            print("預期輸出：")
            print(case["expected"], end="")

    success = passed == len(cases)
    _RESULTS[(qid, language)] = success

    if success:
        print(f"✅ {qid} {label} 全部通過")
    else:
        print(f"❌ {qid} {label}：通過 {passed} / {len(cases)} 個測試")

def _final_report():
    passed_questions = 0
    c_passed = 0
    cpp_passed = 0
    untouched = []

    for i in range(1, 101):
        qid = f"Q{i}"
        c_ok = _RESULTS.get((qid, "c"), False)
        cpp_ok = _RESULTS.get((qid, "cpp"), False)

        if c_ok:
            c_passed += 1
        if cpp_ok:
            cpp_passed += 1
        if c_ok or cpp_ok:
            passed_questions += 1
        if (qid, "c") not in _RESULTS and (qid, "cpp") not in _RESULTS:
            untouched.append(qid)

    print(f"總成績：{passed_questions} / 100 題至少有一種語言通過")
    print(f"C 通過：{c_passed} 題")
    print(f"C++ 通過：{cpp_passed} 題")

    if untouched:
        print("尚未執行任何檢查：" + ", ".join(untouched))

print("C / C++ 自動檢查工具已載入。")
print("作答後不需要手動輸入測試資料。")


## 作答方式提醒

每題流程：

```text
題目
↓
C 或 C++ 作答 cell
↓
檢查 cell
↓
自動送入測試資料
↓
✅ 通過 / ❌ 顯示差異
```

你只需要改 `main()` 內的程式內容。  
**不要修改 `%%c_answer Q...`、`%%cpp_answer Q...` 或檢查 cell。**


---

# Part A：變數操作與公式翻譯

## Q1. 三個整數循環交換

讀入三個整數 a、b、c，使用額外變數完成循環交換，使新 a 等於原 b、新 b 等於原 c、新 c 等於原 a。

**輸入：** 三個整數 a、b、c。

**輸出：** 交換後的 a、b、c，以空格分隔。

**範例輸入：**

```text
10 20 30
```

**範例輸出：**

```text
20 30 10
```

### C 作答

In [ ]:
%%c_answer Q1
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q1 C


In [ ]:
# 檢查 Q1 C：完成上方程式後執行
_run_check("Q1", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q1
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q1 C++


In [ ]:
# 檢查 Q1 C++：完成上方程式後執行
_run_check("Q1", "cpp")


## Q2. 等差數列第 n 項

讀入首項 a、公差 d 與正整數 n，計算等差數列第 n 項。

**輸入：** 三個整數 a、d、n。

**輸出：** 第 n 項。

**範例輸入：**

```text
7 4 12
```

**範例輸出：**

```text
51
```

### C 作答

In [ ]:
%%c_answer Q2
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q2 C


In [ ]:
# 檢查 Q2 C：完成上方程式後執行
_run_check("Q2", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q2
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q2 C++


In [ ]:
# 檢查 Q2 C++：完成上方程式後執行
_run_check("Q2", "cpp")


## Q3. 等差數列前 n 項和

讀入首項 a、公差 d 與正整數 n，計算前 n 項總和。

**輸入：** 三個整數 a、d、n。

**輸出：** 前 n 項和。

**範例輸入：**

```text
3 5 20
```

**範例輸出：**

```text
1010
```

### C 作答

In [ ]:
%%c_answer Q3
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q3 C


In [ ]:
# 檢查 Q3 C：完成上方程式後執行
_run_check("Q3", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q3
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q3 C++


In [ ]:
# 檢查 Q3 C++：完成上方程式後執行
_run_check("Q3", "cpp")


## Q4. 多項式求值

讀入整數 a、b、c、d、x，計算 ax³ + bx² + cx + d。

**輸入：** 五個整數 a、b、c、d、x。

**輸出：** 多項式的值。

**範例輸入：**

```text
2 -3 4 5 6
```

**範例輸出：**

```text
353
```

### C 作答

In [ ]:
%%c_answer Q4
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q4 C


In [ ]:
# 檢查 Q4 C：完成上方程式後執行
_run_check("Q4", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q4
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q4 C++


In [ ]:
# 檢查 Q4 C++：完成上方程式後執行
_run_check("Q4", "cpp")


## Q5. 五科加權成績

讀入五科成績與其權重，計算加權平均。所有權重總和保證大於 0。

**輸入：** 先輸入五個成績，再輸入五個權重，皆為浮點數。

**輸出：** 加權平均，輸出到小數點後 2 位。

**範例輸入：**

```text
80 75 92 88 95
1 1 2 2 4
```

**範例輸出：**

```text
89.50
```

### C 作答

In [ ]:
%%c_answer Q5
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q5 C


In [ ]:
# 檢查 Q5 C：完成上方程式後執行
_run_check("Q5", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q5
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q5 C++


In [ ]:
# 檢查 Q5 C++：完成上方程式後執行
_run_check("Q5", "cpp")


## Q6. 兩數調和平均

讀入兩個正浮點數，計算調和平均。

**輸入：** 兩個正浮點數 a、b。

**輸出：** 調和平均，輸出到小數點後 4 位。

**範例輸入：**

```text
12 18
```

**範例輸出：**

```text
14.4000
```

### C 作答

In [ ]:
%%c_answer Q6
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q6 C


In [ ]:
# 檢查 Q6 C：完成上方程式後執行
_run_check("Q6", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q6
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q6 C++


In [ ]:
# 檢查 Q6 C++：完成上方程式後執行
_run_check("Q6", "cpp")


## Q7. 三數幾何平均

讀入三個正浮點數，計算幾何平均。

**輸入：** 三個正浮點數。

**輸出：** 幾何平均，輸出到小數點後 3 位。

**範例輸入：**

```text
2 8 32
```

**範例輸出：**

```text
8.000
```

### C 作答

In [ ]:
%%c_answer Q7
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q7 C


In [ ]:
# 檢查 Q7 C：完成上方程式後執行
_run_check("Q7", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q7
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q7 C++


In [ ]:
# 檢查 Q7 C++：完成上方程式後執行
_run_check("Q7", "cpp")


## Q8. 比例內分點

點 A(x1,y1) 與 B(x2,y2) 由點 P 以 m:n 內分，AP:PB=m:n。計算 P 座標。

**輸入：** x1、y1、x2、y2、m、n，皆為浮點數。

**輸出：** P 的 x、y，輸出到小數點後 2 位。

**範例輸入：**

```text
1 2 11 17 2 3
```

**範例輸出：**

```text
5.00 8.00
```

### C 作答

In [ ]:
%%c_answer Q8
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q8 C


In [ ]:
# 檢查 Q8 C：完成上方程式後執行
_run_check("Q8", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q8
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q8 C++


In [ ]:
# 檢查 Q8 C++：完成上方程式後執行
_run_check("Q8", "cpp")


## Q9. 二次式判別式

讀入二次式 ax²+bx+c 的三個係數，計算判別式 b²−4ac。

**輸入：** 三個整數 a、b、c。

**輸出：** 判別式。

**範例輸入：**

```text
3 -10 4
```

**範例輸出：**

```text
52
```

### C 作答

In [ ]:
%%c_answer Q9
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q9 C


In [ ]:
# 檢查 Q9 C：完成上方程式後執行
_run_check("Q9", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q9
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q9 C++


In [ ]:
# 檢查 Q9 C++：完成上方程式後執行
_run_check("Q9", "cpp")


## Q10. 二次方程兩根

讀入 a、b、c，保證 a 不為 0 且判別式為正，輸出較大根與較小根。

**輸入：** 三個浮點數 a、b、c。

**輸出：** 較大根與較小根，各到小數點後 3 位。

**範例輸入：**

```text
1 -5 6
```

**範例輸出：**

```text
3.000 2.000
```

### C 作答

In [ ]:
%%c_answer Q10
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q10 C


In [ ]:
# 檢查 Q10 C：完成上方程式後執行
_run_check("Q10", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q10
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q10 C++


In [ ]:
# 檢查 Q10 C++：完成上方程式後執行
_run_check("Q10", "cpp")


---

# Part B：整數拆解、商與餘數

## Q11. 反轉三位數

讀入一個 100 到 999 的整數，輸出反轉後的數字。

**輸入：** 一個三位正整數。

**輸出：** 反轉後的整數；前導零不保留。

**範例輸入：**

```text
470
```

**範例輸出：**

```text
74
```

### C 作答

In [ ]:
%%c_answer Q11
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q11 C


In [ ]:
# 檢查 Q11 C：完成上方程式後執行
_run_check("Q11", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q11
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q11 C++


In [ ]:
# 檢查 Q11 C++：完成上方程式後執行
_run_check("Q11", "cpp")


## Q12. 四位數各位和與積

讀入四位正整數，計算四個數字的總和與乘積。

**輸入：** 一個四位正整數。

**輸出：** 先輸出總和，再輸出乘積。

**範例輸入：**

```text
2307
```

**範例輸出：**

```text
12 0
```

### C 作答

In [ ]:
%%c_answer Q12
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q12 C


In [ ]:
# 檢查 Q12 C：完成上方程式後執行
_run_check("Q12", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q12
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q12 C++


In [ ]:
# 檢查 Q12 C++：完成上方程式後執行
_run_check("Q12", "cpp")


## Q13. 重組六位數

讀入六個 0 到 9 的數字，依輸入順序組成六位整數。第一個數字不為 0。

**輸入：** 六個整數數字。

**輸出：** 組成後的整數。

**範例輸入：**

```text
9 0 2 6 1 4
```

**範例輸出：**

```text
902614
```

### C 作答

In [ ]:
%%c_answer Q13
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q13 C


In [ ]:
# 檢查 Q13 C：完成上方程式後執行
_run_check("Q13", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q13
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q13 C++


In [ ]:
# 檢查 Q13 C++：完成上方程式後執行
_run_check("Q13", "cpp")


## Q14. 交換四位數中間兩位

讀入四位正整數 abcd，輸出 acbd。

**輸入：** 一個四位正整數。

**輸出：** 交換中間兩位後的整數。

**範例輸入：**

```text
5831
```

**範例輸出：**

```text
5381
```

### C 作答

In [ ]:
%%c_answer Q14
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q14 C


In [ ]:
# 檢查 Q14 C：完成上方程式後執行
_run_check("Q14", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q14
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q14 C++


In [ ]:
# 檢查 Q14 C++：完成上方程式後執行
_run_check("Q14", "cpp")


## Q15. 五位數首尾互換

讀入五位正整數 abcde，輸出 ebcda。

**輸入：** 一個五位正整數，最後一位不為 0。

**輸出：** 首尾互換後的整數。

**範例輸入：**

```text
31427
```

**範例輸出：**

```text
71423
```

### C 作答

In [ ]:
%%c_answer Q15
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q15 C


In [ ]:
# 檢查 Q15 C：完成上方程式後執行
_run_check("Q15", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q15
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q15 C++


In [ ]:
# 檢查 Q15 C++：完成上方程式後執行
_run_check("Q15", "cpp")


## Q16. 秒數拆成時分秒

讀入非負總秒數，轉換成小時、分鐘、秒。

**輸入：** 一個非負整數秒數。

**輸出：** 小時、分鐘、秒，以冒號分隔。

**範例輸入：**

```text
10025
```

**範例輸出：**

```text
2:47:5
```

### C 作答

In [ ]:
%%c_answer Q16
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q16 C


In [ ]:
# 檢查 Q16 C：完成上方程式後執行
_run_check("Q16", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q16
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q16 C++


In [ ]:
# 檢查 Q16 C++：完成上方程式後執行
_run_check("Q16", "cpp")


## Q17. 毫秒拆解

讀入非負總毫秒數，轉換成小時、分鐘、秒與毫秒。

**輸入：** 一個非負整數毫秒數。

**輸出：** h m s ms，以空格分隔。

**範例輸入：**

```text
3723456
```

**範例輸出：**

```text
1 2 3 456
```

### C 作答

In [ ]:
%%c_answer Q17
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q17 C


In [ ]:
# 檢查 Q17 C：完成上方程式後執行
_run_check("Q17", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q17
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q17 C++


In [ ]:
# 檢查 Q17 C++：完成上方程式後執行
_run_check("Q17", "cpp")


## Q18. 分鐘拆成日時分

讀入總分鐘數，轉換成天、小時、分鐘。

**輸入：** 一個非負整數分鐘數。

**輸出：** 天、小時、分鐘。

**範例輸入：**

```text
3500
```

**範例輸出：**

```text
2 10 20
```

### C 作答

In [ ]:
%%c_answer Q18
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q18 C


In [ ]:
# 檢查 Q18 C：完成上方程式後執行
_run_check("Q18", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q18
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q18 C++


In [ ]:
# 檢查 Q18 C++：完成上方程式後執行
_run_check("Q18", "cpp")


## Q19. 未來時刻

讀入目前的小時、分鐘，以及經過的分鐘數，計算 24 小時制的新時刻。

**輸入：** 整數 h、m、passed；0≤h<24、0≤m<60。

**輸出：** 新時刻的小時與分鐘，以空格分隔。

**範例輸入：**

```text
23 50 135
```

**範例輸出：**

```text
2 5
```

### C 作答

In [ ]:
%%c_answer Q19
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q19 C


In [ ]:
# 檢查 Q19 C：完成上方程式後執行
_run_check("Q19", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q19
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q19 C++


In [ ]:
# 檢查 Q19 C++：完成上方程式後執行
_run_check("Q19", "cpp")


## Q20. 未來星期索引

以 0 到 6 表示星期日到星期六。讀入今天索引與經過天數，輸出未來星期索引。

**輸入：** 兩個非負整數 today、days。

**輸出：** 未來星期索引。

**範例輸入：**

```text
5 100
```

**範例輸出：**

```text
0
```

### C 作答

In [ ]:
%%c_answer Q20
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q20 C


In [ ]:
# 檢查 Q20 C：完成上方程式後執行
_run_check("Q20", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q20
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q20 C++


In [ ]:
# 檢查 Q20 C++：完成上方程式後執行
_run_check("Q20", "cpp")


## Q21. 平均分糖果

將 candies 顆糖平均分給 students 位學生，輸出每人數量與剩餘數量。

**輸入：** 兩個正整數 candies、students。

**輸出：** 每人數量與剩餘數量。

**範例輸入：**

```text
137 12
```

**範例輸出：**

```text
11 5
```

### C 作答

In [ ]:
%%c_answer Q21
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q21 C


In [ ]:
# 檢查 Q21 C：完成上方程式後執行
_run_check("Q21", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q21
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q21 C++


In [ ]:
# 檢查 Q21 C++：完成上方程式後執行
_run_check("Q21", "cpp")


## Q22. 硬幣最少枚數分解

讀入 0 到 999 的金額，依序使用 50、10、5、1 元硬幣分解。

**輸入：** 一個非負整數 amount。

**輸出：** 50、10、5、1 元硬幣枚數。

**範例輸入：**

```text
287
```

**範例輸出：**

```text
5 3 1 2
```

### C 作答

In [ ]:
%%c_answer Q22
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q22 C


In [ ]:
# 檢查 Q22 C：完成上方程式後執行
_run_check("Q22", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q22
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q22 C++


In [ ]:
# 檢查 Q22 C++：完成上方程式後執行
_run_check("Q22", "cpp")


## Q23. 紙鈔與硬幣分解

讀入正整數金額，依序分解為 1000、500、100、50、10、5、1 元。

**輸入：** 一個非負整數 amount。

**輸出：** 七種面額的張數或枚數。

**範例輸入：**

```text
3686
```

**範例輸出：**

```text
3 1 1 1 3 1 1
```

### C 作答

In [ ]:
%%c_answer Q23
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q23 C


In [ ]:
# 檢查 Q23 C：完成上方程式後執行
_run_check("Q23", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q23
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q23 C++


In [ ]:
# 檢查 Q23 C++：完成上方程式後執行
_run_check("Q23", "cpp")


## Q24. 頁碼分組

一本書有 pages 頁，每冊最多裝 capacity 頁。輸出完整冊數與最後剩餘頁數。

**輸入：** 兩個正整數 pages、capacity。

**輸出：** 完整冊數與剩餘頁數。

**範例輸入：**

```text
1234 120
```

**範例輸出：**

```text
10 34
```

### C 作答

In [ ]:
%%c_answer Q24
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q24 C


In [ ]:
# 檢查 Q24 C：完成上方程式後執行
_run_check("Q24", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q24
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q24 C++


In [ ]:
# 檢查 Q24 C++：完成上方程式後執行
_run_check("Q24", "cpp")


## Q25. 資料區塊分割

一個檔案有 bytes 位元組，每個區塊 block 位元組。輸出完整區塊數與剩餘位元組。

**輸入：** 兩個正整數 bytes、block。

**輸出：** 完整區塊數與剩餘位元組。

**範例輸入：**

```text
1048579 4096
```

**範例輸出：**

```text
256 3
```

### C 作答

In [ ]:
%%c_answer Q25
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q25 C


In [ ]:
# 檢查 Q25 C：完成上方程式後執行
_run_check("Q25", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q25
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q25 C++


In [ ]:
# 檢查 Q25 C++：完成上方程式後執行
_run_check("Q25", "cpp")


## Q26. 循環座位編號

座位編號從 1 到 n 循環。讀入目前座位 current 與往後移動 steps 格，輸出新座位。

**輸入：** 三個正整數 n、current、steps。

**輸出：** 新的 1-based 座位編號。

**範例輸入：**

```text
12 10 17
```

**範例輸出：**

```text
3
```

### C 作答

In [ ]:
%%c_answer Q26
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q26 C


In [ ]:
# 檢查 Q26 C：完成上方程式後執行
_run_check("Q26", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q26
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q26 C++


In [ ]:
# 檢查 Q26 C++：完成上方程式後執行
_run_check("Q26", "cpp")


## Q27. 循環陣列反向位移索引

索引從 0 到 n−1。讀入目前索引 current 與向左位移 steps，且 steps 不超過 current+n，輸出新索引。

**輸入：** 三個整數 n、current、steps。

**輸出：** 新索引。

**範例輸入：**

```text
10 3 7
```

**範例輸出：**

```text
6
```

### C 作答

In [ ]:
%%c_answer Q27
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q27 C


In [ ]:
# 檢查 Q27 C：完成上方程式後執行
_run_check("Q27", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q27
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q27 C++


In [ ]:
# 檢查 Q27 C++：完成上方程式後執行
_run_check("Q27", "cpp")


## Q28. 九宮格位置編碼

一個 3×3 格子使用 0 到 8 的一維索引。輸入索引，輸出列與欄，皆從 0 開始。

**輸入：** 一個 0 到 8 的整數 index。

**輸出：** row column。

**範例輸入：**

```text
7
```

**範例輸出：**

```text
2 1
```

### C 作答

In [ ]:
%%c_answer Q28
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q28 C


In [ ]:
# 檢查 Q28 C：完成上方程式後執行
_run_check("Q28", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q28
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q28 C++


In [ ]:
# 檢查 Q28 C++：完成上方程式後執行
_run_check("Q28", "cpp")


## Q29. 矩形網格索引

一個有 cols 欄的網格使用 row-major 一維索引。輸入 row、col、cols，求一維索引。

**輸入：** 三個非負整數 row、col、cols。

**輸出：** 一維索引。

**範例輸入：**

```text
7 11 20
```

**範例輸出：**

```text
151
```

### C 作答

In [ ]:
%%c_answer Q29
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q29 C


In [ ]:
# 檢查 Q29 C：完成上方程式後執行
_run_check("Q29", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q29
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q29 C++


In [ ]:
# 檢查 Q29 C++：完成上方程式後執行
_run_check("Q29", "cpp")


## Q30. RGB 整數編碼

讀入 0 到 255 的 R、G、B，將它們編碼成 R×65536 + G×256 + B。

**輸入：** 三個整數 R、G、B。

**輸出：** 編碼後的十進位整數。

**範例輸入：**

```text
12 200 45
```

**範例輸出：**

```text
837677
```

### C 作答

In [ ]:
%%c_answer Q30
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q30 C


In [ ]:
# 檢查 Q30 C：完成上方程式後執行
_run_check("Q30", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q30
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q30 C++


In [ ]:
# 檢查 Q30 C++：完成上方程式後執行
_run_check("Q30", "cpp")


## Q31. 解碼 RGB 整數

讀入 0 到 16777215 的整數編碼，解碼 R、G、B。

**輸入：** 一個非負整數 code。

**輸出：** R G B。

**範例輸入：**

```text
837677
```

**範例輸出：**

```text
12 200 45
```

### C 作答

In [ ]:
%%c_answer Q31
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q31 C


In [ ]:
# 檢查 Q31 C：完成上方程式後執行
_run_check("Q31", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q31
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q31 C++


In [ ]:
# 檢查 Q31 C++：完成上方程式後執行
_run_check("Q31", "cpp")


---

# Part C：型別轉換與精度

## Q32. 整數比例

讀入兩個整數 a、b，保證 b 不為 0，計算 a/b 的實數結果。

**輸入：** 兩個整數。

**輸出：** 結果到小數點後 5 位。

**範例輸入：**

```text
7 12
```

**範例輸出：**

```text
0.58333
```

### C 作答

In [ ]:
%%c_answer Q32
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q32 C


In [ ]:
# 檢查 Q32 C：完成上方程式後執行
_run_check("Q32", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q32
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q32 C++


In [ ]:
# 檢查 Q32 C++：完成上方程式後執行
_run_check("Q32", "cpp")


## Q33. 五個整數平均

讀入五個整數，輸出精確平均值。

**輸入：** 五個整數。

**輸出：** 平均到小數點後 2 位。

**範例輸入：**

```text
10 11 12 13 17
```

**範例輸出：**

```text
12.60
```

### C 作答

In [ ]:
%%c_answer Q33
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q33 C


In [ ]:
# 檢查 Q33 C：完成上方程式後執行
_run_check("Q33", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q33
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q33 C++


In [ ]:
# 檢查 Q33 C++：完成上方程式後執行
_run_check("Q33", "cpp")


## Q34. 截斷與小數部分

讀入一個非負浮點數 x，輸出截斷後整數部分與小數部分。

**輸入：** 一個非負浮點數。

**輸出：** 整數部分與小數部分；小數部分到 4 位。

**範例輸入：**

```text
123.4567
```

**範例輸出：**

```text
123 0.4567
```

### C 作答

In [ ]:
%%c_answer Q34
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q34 C


In [ ]:
# 檢查 Q34 C：完成上方程式後執行
_run_check("Q34", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q34
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q34 C++


In [ ]:
# 檢查 Q34 C++：完成上方程式後執行
_run_check("Q34", "cpp")


## Q35. 正數四捨五入

讀入一個非負浮點數，使用型別轉換完成四捨五入到整數。

**輸入：** 一個非負浮點數 x。

**輸出：** 四捨五入後的整數。

**範例輸入：**

```text
78.65
```

**範例輸出：**

```text
79
```

### C 作答

In [ ]:
%%c_answer Q35
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q35 C


In [ ]:
# 檢查 Q35 C：完成上方程式後執行
_run_check("Q35", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q35
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q35 C++


In [ ]:
# 檢查 Q35 C++：完成上方程式後執行
_run_check("Q35", "cpp")


## Q36. 百分比成績

讀入答對題數 correct 與總題數 total，計算百分比。

**輸入：** 兩個正整數，correct≤total。

**輸出：** 百分比到小數點後 2 位，後接 %。

**範例輸入：**

```text
37 45
```

**範例輸出：**

```text
82.22%
```

### C 作答

In [ ]:
%%c_answer Q36
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q36 C


In [ ]:
# 檢查 Q36 C：完成上方程式後執行
_run_check("Q36", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q36
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q36 C++


In [ ]:
# 檢查 Q36 C++：完成上方程式後執行
_run_check("Q36", "cpp")


## Q37. 攝氏轉華氏與開爾文

讀入攝氏溫度，輸出華氏與開爾文溫度。

**輸入：** 一個浮點數攝氏溫度。

**輸出：** 華氏、開爾文，各到小數點後 2 位。

**範例輸入：**

```text
36.5
```

**範例輸出：**

```text
97.70 309.65
```

### C 作答

In [ ]:
%%c_answer Q37
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q37 C


In [ ]:
# 檢查 Q37 C：完成上方程式後執行
_run_check("Q37", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q37
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q37 C++


In [ ]:
# 檢查 Q37 C++：完成上方程式後執行
_run_check("Q37", "cpp")


## Q38. 金額拆成元與分

讀入非負金額（浮點數），先四捨五入成總分，再輸出元與分。

**輸入：** 一個非負浮點數 amount。

**輸出：** 整數元與整數分。

**範例輸入：**

```text
123.456
```

**範例輸出：**

```text
123 46
```

### C 作答

In [ ]:
%%c_answer Q38
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q38 C


In [ ]:
# 檢查 Q38 C：完成上方程式後執行
_run_check("Q38", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q38
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q38 C++


In [ ]:
# 檢查 Q38 C++：完成上方程式後執行
_run_check("Q38", "cpp")


## Q39. 十進位小時轉時分

讀入非負十進位小時，例如 2.75 代表 2 小時 45 分。以四捨五入處理分鐘。

**輸入：** 一個非負浮點數 hours。

**輸出：** 整數小時與分鐘。

**範例輸入：**

```text
7.425
```

**範例輸出：**

```text
7 26
```

### C 作答

In [ ]:
%%c_answer Q39
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q39 C


In [ ]:
# 檢查 Q39 C：完成上方程式後執行
_run_check("Q39", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q39
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q39 C++


In [ ]:
# 檢查 Q39 C++：完成上方程式後執行
_run_check("Q39", "cpp")


## Q40. 每公升公里數

讀入行駛公里數與耗油公升數，計算 km/L。

**輸入：** 兩個正浮點數 distance、liters。

**輸出：** km/L 到小數點後 3 位。

**範例輸入：**

```text
612.5 42.8
```

**範例輸出：**

```text
14.311
```

### C 作答

In [ ]:
%%c_answer Q40
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q40 C


In [ ]:
# 檢查 Q40 C：完成上方程式後執行
_run_check("Q40", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q40
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q40 C++


In [ ]:
# 檢查 Q40 C++：完成上方程式後執行
_run_check("Q40", "cpp")


## Q41. 百公里油耗

讀入行駛公里數與耗油公升數，計算 L/100km。

**輸入：** 兩個正浮點數 distance、liters。

**輸出：** L/100km 到小數點後 2 位。

**範例輸入：**

```text
612.5 42.8
```

**範例輸出：**

```text
6.99
```

### C 作答

In [ ]:
%%c_answer Q41
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q41 C


In [ ]:
# 檢查 Q41 C：完成上方程式後執行
_run_check("Q41", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q41
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q41 C++


In [ ]:
# 檢查 Q41 C++：完成上方程式後執行
_run_check("Q41", "cpp")


## Q42. 每秒資料率

讀入總位元組數 bytes 與秒數 seconds，計算 MiB/s；1 MiB=1048576 bytes。

**輸入：** bytes 為 long long，seconds 為正浮點數。

**輸出：** MiB/s 到小數點後 4 位。

**範例輸入：**

```text
9876543210 73.5
```

**範例輸出：**

```text
128.1497
```

### C 作答

In [ ]:
%%c_answer Q42
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q42 C


In [ ]:
# 檢查 Q42 C：完成上方程式後執行
_run_check("Q42", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q42
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q42 C++


In [ ]:
# 檢查 Q42 C++：完成上方程式後執行
_run_check("Q42", "cpp")


## Q43. 浮點線性插值

讀入 a、b 與 t（0≤t≤1），計算 a+(b−a)t。

**輸入：** 三個浮點數。

**輸出：** 插值結果到小數點後 3 位。

**範例輸入：**

```text
10.5 42.0 0.35
```

**範例輸出：**

```text
21.525
```

### C 作答

In [ ]:
%%c_answer Q43
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q43 C


In [ ]:
# 檢查 Q43 C：完成上方程式後執行
_run_check("Q43", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q43
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q43 C++


In [ ]:
# 檢查 Q43 C++：完成上方程式後執行
_run_check("Q43", "cpp")


---

# Part D：字元編碼與循環運算

## Q44. 小寫轉大寫

讀入一個英文小寫字母，利用字元編碼轉成大寫。

**輸入：** 一個小寫英文字母。

**輸出：** 對應的大寫字母。

**範例輸入：**

```text
q
```

**範例輸出：**

```text
Q
```

### C 作答

In [ ]:
%%c_answer Q44
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q44 C


In [ ]:
# 檢查 Q44 C：完成上方程式後執行
_run_check("Q44", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q44
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q44 C++


In [ ]:
# 檢查 Q44 C++：完成上方程式後執行
_run_check("Q44", "cpp")


## Q45. 大寫轉小寫

讀入一個英文大寫字母，利用字元編碼轉成小寫。

**輸入：** 一個大寫英文字母。

**輸出：** 對應的小寫字母。

**範例輸入：**

```text
M
```

**範例輸出：**

```text
m
```

### C 作答

In [ ]:
%%c_answer Q45
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q45 C


In [ ]:
# 檢查 Q45 C：完成上方程式後執行
_run_check("Q45", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q45
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q45 C++


In [ ]:
# 檢查 Q45 C++：完成上方程式後執行
_run_check("Q45", "cpp")


## Q46. 字母順位

讀入一個大寫英文字母，輸出它在字母表中的 1-based 順位。

**輸入：** 一個大寫字母。

**輸出：** 1 到 26 的整數。

**範例輸入：**

```text
T
```

**範例輸出：**

```text
20
```

### C 作答

In [ ]:
%%c_answer Q46
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q46 C


In [ ]:
# 檢查 Q46 C：完成上方程式後執行
_run_check("Q46", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q46
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q46 C++


In [ ]:
# 檢查 Q46 C++：完成上方程式後執行
_run_check("Q46", "cpp")


## Q47. 順位轉大寫字母

讀入 1 到 26 的整數，輸出對應的大寫字母。

**輸入：** 一個 1 到 26 的整數。

**輸出：** 大寫字母。

**範例輸入：**

```text
22
```

**範例輸出：**

```text
V
```

### C 作答

In [ ]:
%%c_answer Q47
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q47 C


In [ ]:
# 檢查 Q47 C：完成上方程式後執行
_run_check("Q47", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q47
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q47 C++


In [ ]:
# 檢查 Q47 C++：完成上方程式後執行
_run_check("Q47", "cpp")


## Q48. 凱撒位移大寫字母

讀入大寫字母與非負位移 k，字母超過 Z 時回到 A。

**輸入：** 一個大寫字母 ch 與整數 k。

**輸出：** 位移後的大寫字母。

**範例輸入：**

```text
X 31
```

**範例輸出：**

```text
C
```

### C 作答

In [ ]:
%%c_answer Q48
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q48 C


In [ ]:
# 檢查 Q48 C：完成上方程式後執行
_run_check("Q48", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q48
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q48 C++


In [ ]:
# 檢查 Q48 C++：完成上方程式後執行
_run_check("Q48", "cpp")


## Q49. 數字字元轉整數

讀入一個 0 到 9 的數字字元，轉成對應整數並計算其平方。

**輸入：** 一個數字字元。

**輸出：** 整數值與平方。

**範例輸入：**

```text
7
```

**範例輸出：**

```text
7 49
```

### C 作答

In [ ]:
%%c_answer Q49
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q49 C


In [ ]:
# 檢查 Q49 C：完成上方程式後執行
_run_check("Q49", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q49
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q49 C++


In [ ]:
# 檢查 Q49 C++：完成上方程式後執行
_run_check("Q49", "cpp")


## Q50. 整數轉數字字元

讀入 0 到 9 的整數，轉成對應數字字元，並輸出下一個 ASCII 字元。輸入保證小於 9。

**輸入：** 一個 0 到 8 的整數。

**輸出：** 目前數字字元與下一個數字字元。

**範例輸入：**

```text
5
```

**範例輸出：**

```text
5 6
```

### C 作答

In [ ]:
%%c_answer Q50
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q50 C


In [ ]:
# 檢查 Q50 C：完成上方程式後執行
_run_check("Q50", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q50
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q50 C++


In [ ]:
# 檢查 Q50 C++：完成上方程式後執行
_run_check("Q50", "cpp")


## Q51. 兩字母距離

讀入兩個大寫字母，輸出第二個字母編碼減第一個字母編碼。保證第二個不小於第一個。

**輸入：** 兩個大寫字母。

**輸出：** 非負距離。

**範例輸入：**

```text
F R
```

**範例輸出：**

```text
12
```

### C 作答

In [ ]:
%%c_answer Q51
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q51 C


In [ ]:
# 檢查 Q51 C：完成上方程式後執行
_run_check("Q51", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q51
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q51 C++


In [ ]:
# 檢查 Q51 C++：完成上方程式後執行
_run_check("Q51", "cpp")


## Q52. 雙字母 0-based 編碼

將兩個大寫字母視為 26 進位的兩位數：AA=0、AB=1、BA=26。

**輸入：** 兩個大寫字母。

**輸出：** 0-based 編碼值。

**範例輸入：**

```text
D K
```

**範例輸出：**

```text
88
```

### C 作答

In [ ]:
%%c_answer Q52
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q52 C


In [ ]:
# 檢查 Q52 C：完成上方程式後執行
_run_check("Q52", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q52
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q52 C++


In [ ]:
# 檢查 Q52 C++：完成上方程式後執行
_run_check("Q52", "cpp")


## Q53. 解碼雙字母

讀入 0 到 675 的整數，依 AA=0、AB=1、BA=26 的規則解碼。

**輸入：** 一個 0 到 675 的整數 code。

**輸出：** 兩個大寫字母。

**範例輸入：**

```text
347
```

**範例輸出：**

```text
NJ
```

### C 作答

In [ ]:
%%c_answer Q53
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q53 C


In [ ]:
# 檢查 Q53 C：完成上方程式後執行
_run_check("Q53", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q53
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q53 C++


In [ ]:
# 檢查 Q53 C++：完成上方程式後執行
_run_check("Q53", "cpp")


---

# Part E：幾何與科學計算

## Q54. 長方形完整資料

讀入長與寬，計算周長、面積與對角線。

**輸入：** 兩個正浮點數 length、width。

**輸出：** 周長、面積、對角線，各到小數點後 2 位。

**範例輸入：**

```text
8.4 5.7
```

**範例輸出：**

```text
28.20 47.88 10.15
```

### C 作答

In [ ]:
%%c_answer Q54
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q54 C


In [ ]:
# 檢查 Q54 C：完成上方程式後執行
_run_check("Q54", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q54
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q54 C++


In [ ]:
# 檢查 Q54 C++：完成上方程式後執行
_run_check("Q54", "cpp")


## Q55. 海龍公式三角形面積

讀入可構成三角形的三邊，使用海龍公式計算面積。

**輸入：** 三個正浮點數 a、b、c。

**輸出：** 面積到小數點後 3 位。

**範例輸入：**

```text
7.5 8.2 9.1
```

**範例輸出：**

```text
29.020
```

### C 作答

In [ ]:
%%c_answer Q55
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q55 C


In [ ]:
# 檢查 Q55 C：完成上方程式後執行
_run_check("Q55", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q55
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q55 C++


In [ ]:
# 檢查 Q55 C++：完成上方程式後執行
_run_check("Q55", "cpp")


## Q56. 圓環面積

讀入外半徑 R 與內半徑 r，R>r，計算圓環面積。π 使用 acos(-1)。

**輸入：** 兩個正浮點數 R、r。

**輸出：** 面積到小數點後 4 位。

**範例輸入：**

```text
10 6.5
```

**範例輸出：**

```text
181.4270
```

### C 作答

In [ ]:
%%c_answer Q56
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q56 C


In [ ]:
# 檢查 Q56 C：完成上方程式後執行
_run_check("Q56", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q56
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q56 C++


In [ ]:
# 檢查 Q56 C++：完成上方程式後執行
_run_check("Q56", "cpp")


## Q57. 圓柱表面積與體積

讀入半徑 r 與高 h，計算總表面積與體積。π 使用 acos(-1)。

**輸入：** 兩個正浮點數。

**輸出：** 總表面積與體積，各到小數點後 3 位。

**範例輸入：**

```text
3.2 11.5
```

**範例輸出：**

```text
295.561 369.954
```

### C 作答

In [ ]:
%%c_answer Q57
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q57 C


In [ ]:
# 檢查 Q57 C：完成上方程式後執行
_run_check("Q57", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q57
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q57 C++


In [ ]:
# 檢查 Q57 C++：完成上方程式後執行
_run_check("Q57", "cpp")


## Q58. 圓錐斜高與體積

讀入圓錐底半徑 r 與高 h，計算斜高與體積。

**輸入：** 兩個正浮點數。

**輸出：** 斜高與體積，各到小數點後 3 位。

**範例輸入：**

```text
5.5 12
```

**範例輸出：**

```text
13.200 380.133
```

### C 作答

In [ ]:
%%c_answer Q58
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q58 C


In [ ]:
# 檢查 Q58 C：完成上方程式後執行
_run_check("Q58", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q58
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q58 C++


In [ ]:
# 檢查 Q58 C++：完成上方程式後執行
_run_check("Q58", "cpp")


## Q59. 兩點距離與中點

讀入平面上兩點座標，計算距離與中點。

**輸入：** 四個浮點數 x1、y1、x2、y2。

**輸出：** 距離、midX、midY，各到小數點後 3 位。

**範例輸入：**

```text
-2.5 4 8.5 -3
```

**範例輸出：**

```text
13.038 3.000 0.500
```

### C 作答

In [ ]:
%%c_answer Q59
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q59 C


In [ ]:
# 檢查 Q59 C：完成上方程式後執行
_run_check("Q59", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q59
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q59 C++


In [ ]:
# 檢查 Q59 C++：完成上方程式後執行
_run_check("Q59", "cpp")


## Q60. 三維空間距離

讀入三維空間中兩點座標，計算距離。

**輸入：** 六個浮點數。

**輸出：** 距離到小數點後 4 位。

**範例輸入：**

```text
1 2 3 7 -2 11
```

**範例輸出：**

```text
10.7703
```

### C 作答

In [ ]:
%%c_answer Q60
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q60 C


In [ ]:
# 檢查 Q60 C：完成上方程式後執行
_run_check("Q60", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q60
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q60 C++


In [ ]:
# 檢查 Q60 C++：完成上方程式後執行
_run_check("Q60", "cpp")


## Q61. 斜率與截距

讀入兩點座標，保證 x1≠x2，求通過兩點直線的斜率 m 與 y 截距 b。

**輸入：** 四個浮點數。

**輸出：** m 與 b，各到小數點後 3 位。

**範例輸入：**

```text
2 5 8 23
```

**範例輸出：**

```text
3.000 -1.000
```

### C 作答

In [ ]:
%%c_answer Q61
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q61 C


In [ ]:
# 檢查 Q61 C：完成上方程式後執行
_run_check("Q61", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q61
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q61 C++


In [ ]:
# 檢查 Q61 C++：完成上方程式後執行
_run_check("Q61", "cpp")


## Q62. 向量內積與長度乘積

讀入二維向量 (ax,ay)、(bx,by)，計算內積與兩向量長度乘積。

**輸入：** 四個浮點數。

**輸出：** 內積與長度乘積，各到小數點後 3 位。

**範例輸入：**

```text
3 4 -2 5
```

**範例輸出：**

```text
14.000 26.926
```

### C 作答

In [ ]:
%%c_answer Q62
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q62 C


In [ ]:
# 檢查 Q62 C：完成上方程式後執行
_run_check("Q62", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q62
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q62 C++


In [ ]:
# 檢查 Q62 C++：完成上方程式後執行
_run_check("Q62", "cpp")


## Q63. 三角形重心

讀入三個頂點座標，計算重心。

**輸入：** 六個浮點數。

**輸出：** 重心 x、y，各到小數點後 3 位。

**範例輸入：**

```text
0 0 9 3 3 12
```

**範例輸出：**

```text
4.000 5.000
```

### C 作答

In [ ]:
%%c_answer Q63
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q63 C


In [ ]:
# 檢查 Q63 C：完成上方程式後執行
_run_check("Q63", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q63
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q63 C++


In [ ]:
# 檢查 Q63 C++：完成上方程式後執行
_run_check("Q63", "cpp")


## Q64. BMI 與理想體重差

讀入體重 kg、身高 cm 與目標 BMI，計算目前 BMI 與達到目標 BMI 所需的體重差（目標體重−目前體重）。

**輸入：** 三個正浮點數。

**輸出：** 目前 BMI 與體重差，各到小數點後 2 位。

**範例輸入：**

```text
72.5 176 22
```

**範例輸出：**

```text
23.41 -4.35
```

### C 作答

In [ ]:
%%c_answer Q64
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q64 C


In [ ]:
# 檢查 Q64 C：完成上方程式後執行
_run_check("Q64", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q64
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q64 C++


In [ ]:
# 檢查 Q64 C++：完成上方程式後執行
_run_check("Q64", "cpp")


## Q65. 動能與動量

讀入質量 m 與速度 v，計算動量 p=mv 與動能 E=mv²/2。

**輸入：** 兩個浮點數。

**輸出：** 動量與動能，各到小數點後 3 位。

**範例輸入：**

```text
12.5 7.2
```

**範例輸出：**

```text
90.000 324.000
```

### C 作答

In [ ]:
%%c_answer Q65
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q65 C


In [ ]:
# 檢查 Q65 C：完成上方程式後執行
_run_check("Q65", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q65
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q65 C++


In [ ]:
# 檢查 Q65 C++：完成上方程式後執行
_run_check("Q65", "cpp")


## Q66. 位能與自由落下末速

讀入質量 m 與高度 h，取 g=9.80665，計算重力位能 mgh 與由靜止落下的理論末速 sqrt(2gh)。

**輸入：** 兩個正浮點數。

**輸出：** 位能與末速，各到小數點後 3 位。

**範例輸入：**

```text
2.4 18.5
```

**範例輸出：**

```text
435.415 19.049
```

### C 作答

In [ ]:
%%c_answer Q66
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q66 C


In [ ]:
# 檢查 Q66 C：完成上方程式後執行
_run_check("Q66", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q66
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q66 C++


In [ ]:
# 檢查 Q66 C++：完成上方程式後執行
_run_check("Q66", "cpp")


## Q67. 平拋落地時間與水平距離

物體從高度 h 以水平速度 vx 射出，取 g=9.80665，計算落地時間與水平距離。

**輸入：** 兩個正浮點數 h、vx。

**輸出：** 時間與距離，各到小數點後 3 位。

**範例輸入：**

```text
45 18
```

**範例輸出：**

```text
3.029 54.530
```

### C 作答

In [ ]:
%%c_answer Q67
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q67 C


In [ ]:
# 檢查 Q67 C：完成上方程式後執行
_run_check("Q67", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q67
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q67 C++


In [ ]:
# 檢查 Q67 C++：完成上方程式後執行
_run_check("Q67", "cpp")


## Q68. 拋射運動射程

讀入初速 v 與角度 degree，取 g=9.80665，計算同高度落地的理論射程 v²sin(2θ)/g。

**輸入：** 兩個正浮點數 v、degree。

**輸出：** 射程到小數點後 3 位。

**範例輸入：**

```text
30 38
```

**範例輸出：**

```text
89.048
```

### C 作答

In [ ]:
%%c_answer Q68
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q68 C


In [ ]:
# 檢查 Q68 C：完成上方程式後執行
_run_check("Q68", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q68
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q68 C++


In [ ]:
# 檢查 Q68 C++：完成上方程式後執行
_run_check("Q68", "cpp")


## Q69. 並聯電阻

讀入三個正電阻 R1、R2、R3，計算並聯等效電阻。

**輸入：** 三個正浮點數。

**輸出：** 等效電阻到小數點後 4 位。

**範例輸入：**

```text
100 220 330
```

**範例輸出：**

```text
56.8966
```

### C 作答

In [ ]:
%%c_answer Q69
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q69 C


In [ ]:
# 檢查 Q69 C：完成上方程式後執行
_run_check("Q69", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q69
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q69 C++


In [ ]:
# 檢查 Q69 C++：完成上方程式後執行
_run_check("Q69", "cpp")


## Q70. 歐姆定律與功率

讀入電壓 V 與電阻 R，計算電流 I=V/R 與功率 P=VI。

**輸入：** 兩個正浮點數。

**輸出：** 電流與功率，各到小數點後 3 位。

**範例輸入：**

```text
12 8.2
```

**範例輸出：**

```text
1.463 17.561
```

### C 作答

In [ ]:
%%c_answer Q70
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q70 C


In [ ]:
# 檢查 Q70 C：完成上方程式後執行
_run_check("Q70", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q70
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q70 C++


In [ ]:
# 檢查 Q70 C++：完成上方程式後執行
_run_check("Q70", "cpp")


## Q71. 理想氣體壓力

讀入莫耳數 n、溫度 T(K)、體積 V(L)，使用 R=0.082057 計算 P=nRT/V。

**輸入：** 三個正浮點數。

**輸出：** 壓力到小數點後 4 位。

**範例輸入：**

```text
2.5 310 18
```

**範例輸出：**

```text
3.5330
```

### C 作答

In [ ]:
%%c_answer Q71
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q71 C


In [ ]:
# 檢查 Q71 C：完成上方程式後執行
_run_check("Q71", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q71
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q71 C++


In [ ]:
# 檢查 Q71 C++：完成上方程式後執行
_run_check("Q71", "cpp")


## Q72. 聲音分貝

讀入正的聲強 I，參考聲強 I0=1e-12，計算 10log10(I/I0)。

**輸入：** 一個正浮點數 I。

**輸出：** 分貝到小數點後 2 位。

**範例輸入：**

```text
0.00001
```

**範例輸出：**

```text
70.00
```

### C 作答

In [ ]:
%%c_answer Q72
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q72 C


In [ ]:
# 檢查 Q72 C：完成上方程式後執行
_run_check("Q72", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q72
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q72 C++


In [ ]:
# 檢查 Q72 C++：完成上方程式後執行
_run_check("Q72", "cpp")


## Q73. 酸鹼值 pH

讀入正的氫離子濃度 H，計算 pH=−log10(H)。

**輸入：** 一個正浮點數 H。

**輸出：** pH 到小數點後 3 位。

**範例輸入：**

```text
0.0000032
```

**範例輸出：**

```text
5.495
```

### C 作答

In [ ]:
%%c_answer Q73
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q73 C


In [ ]:
# 檢查 Q73 C：完成上方程式後執行
_run_check("Q73", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q73
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q73 C++


In [ ]:
# 檢查 Q73 C++：完成上方程式後執行
_run_check("Q73", "cpp")


## Q74. 兩溫度混合

忽略熱損失，讀入兩份相同物質的質量 m1、m2 與溫度 t1、t2，計算平衡溫度。

**輸入：** 四個浮點數，質量皆為正。

**輸出：** 平衡溫度到小數點後 2 位。

**範例輸入：**

```text
2.5 80 4.0 20
```

**範例輸出：**

```text
43.08
```

### C 作答

In [ ]:
%%c_answer Q74
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q74 C


In [ ]:
# 檢查 Q74 C：完成上方程式後執行
_run_check("Q74", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q74
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q74 C++


In [ ]:
# 檢查 Q74 C++：完成上方程式後執行
_run_check("Q74", "cpp")


---

# Part F：商業計算、矩陣與分數

## Q75. 單利本利和

讀入本金 principal、年利率 percent 與年數 years，使用單利計算利息與本利和。

**輸入：** 三個非負浮點數。

**輸出：** 利息與本利和，各到小數點後 2 位。

**範例輸入：**

```text
150000 2.4 3.5
```

**範例輸出：**

```text
12600.00 162600.00
```

### C 作答

In [ ]:
%%c_answer Q75
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q75 C


In [ ]:
# 檢查 Q75 C：完成上方程式後執行
_run_check("Q75", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q75
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q75 C++


In [ ]:
# 檢查 Q75 C++：完成上方程式後執行
_run_check("Q75", "cpp")


## Q76. 複利本利和

讀入本金、年利率百分比與年數，假設每年複利一次，計算本利和。

**輸入：** 本金與利率為浮點數，年數為非負整數。

**輸出：** 本利和到小數點後 2 位。

**範例輸入：**

```text
80000 3.25 6
```

**範例輸出：**

```text
96923.78
```

### C 作答

In [ ]:
%%c_answer Q76
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q76 C


In [ ]:
# 檢查 Q76 C：完成上方程式後執行
_run_check("Q76", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q76
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q76 C++


In [ ]:
# 檢查 Q76 C++：完成上方程式後執行
_run_check("Q76", "cpp")


## Q77. 折扣後加稅

讀入原價、折扣百分比與稅率百分比，先折扣再加稅。

**輸入：** 三個非負浮點數。

**輸出：** 折扣後價格與最終價格，各到小數點後 2 位。

**範例輸入：**

```text
2499 18 5
```

**範例輸出：**

```text
2049.18 2151.64
```

### C 作答

In [ ]:
%%c_answer Q77
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q77 C


In [ ]:
# 檢查 Q77 C：完成上方程式後執行
_run_check("Q77", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q77
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q77 C++


In [ ]:
# 檢查 Q77 C++：完成上方程式後執行
_run_check("Q77", "cpp")


## Q78. 團體帳單分攤

讀入餐費 subtotal、服務費百分比、稅率百分比與人數，依序加服務費與稅後平均分攤。

**輸入：** 前三個為浮點數，人數為正整數。

**輸出：** 總額與每人金額，各到小數點後 2 位。

**範例輸入：**

```text
3860 10 5 7
```

**範例輸出：**

```text
4458.30 636.90
```

### C 作答

In [ ]:
%%c_answer Q78
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q78 C


In [ ]:
# 檢查 Q78 C：完成上方程式後執行
_run_check("Q78", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q78
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q78 C++


In [ ]:
# 檢查 Q78 C++：完成上方程式後執行
_run_check("Q78", "cpp")


## Q79. 外幣雙重換算

讀入 TWD 金額、TWD→USD 匯率（1 TWD 可換多少 USD）與 USD→JPY 匯率，計算 USD 與 JPY。

**輸入：** 三個正浮點數。

**輸出：** USD 與 JPY，各到小數點後 2 位。

**範例輸入：**

```text
25000 0.0308 148.6
```

**範例輸出：**

```text
770.00 114422.00
```

### C 作答

In [ ]:
%%c_answer Q79
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q79 C


In [ ]:
# 檢查 Q79 C：完成上方程式後執行
_run_check("Q79", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q79
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q79 C++


In [ ]:
# 檢查 Q79 C++：完成上方程式後執行
_run_check("Q79", "cpp")


## Q80. 每件商品成本

讀入商品總價、運費、折扣百分比與件數，先折扣商品總價，再加運費並平均。

**輸入：** 前三個為浮點數，件數為正整數。

**輸出：** 訂單總成本與每件成本，各到小數點後 2 位。

**範例輸入：**

```text
7200 180 12.5 24
```

**範例輸出：**

```text
6480.00 270.00
```

### C 作答

In [ ]:
%%c_answer Q80
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q80 C


In [ ]:
# 檢查 Q80 C：完成上方程式後執行
_run_check("Q80", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q80
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q80 C++


In [ ]:
# 檢查 Q80 C++：完成上方程式後執行
_run_check("Q80", "cpp")


## Q81. 固定佣金與比例佣金

讀入銷售額 sales、固定佣金 fixed 與比例 percent，計算總佣金。

**輸入：** 三個非負浮點數。

**輸出：** 比例佣金與總佣金，各到小數點後 2 位。

**範例輸入：**

```text
380000 5000 1.75
```

**範例輸出：**

```text
6650.00 11650.00
```

### C 作答

In [ ]:
%%c_answer Q81
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q81 C


In [ ]:
# 檢查 Q81 C：完成上方程式後執行
_run_check("Q81", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q81
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q81 C++


In [ ]:
# 檢查 Q81 C++：完成上方程式後執行
_run_check("Q81", "cpp")


## Q82. 成本、收入與利潤率

讀入單位成本、售價與數量，計算總收入、總成本、利潤及以成本為基準的利潤率。

**輸入：** 兩個浮點數與一個正整數。

**輸出：** 收入、成本、利潤、利潤率%，各到小數點後 2 位。

**範例輸入：**

```text
38.5 59.9 240
```

**範例輸出：**

```text
14376.00 9240.00 5136.00 55.58
```

### C 作答

In [ ]:
%%c_answer Q82
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q82 C


In [ ]:
# 檢查 Q82 C：完成上方程式後執行
_run_check("Q82", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q82
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q82 C++


In [ ]:
# 檢查 Q82 C++：完成上方程式後執行
_run_check("Q82", "cpp")


## Q83. 二階行列式

讀入 2×2 矩陣 a b / c d，計算行列式 ad−bc。

**輸入：** 四個整數 a、b、c、d。

**輸出：** 行列式。

**範例輸入：**

```text
7 -2 5 9
```

**範例輸出：**

```text
73
```

### C 作答

In [ ]:
%%c_answer Q83
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q83 C


In [ ]:
# 檢查 Q83 C：完成上方程式後執行
_run_check("Q83", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q83
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q83 C++


In [ ]:
# 檢查 Q83 C++：完成上方程式後執行
_run_check("Q83", "cpp")


## Q84. 二乘二矩陣相乘

讀入矩陣 A 的四個元素，再讀入矩陣 B 的四個元素，計算 A×B。

**輸入：** 八個整數，順序為 a11 a12 a21 a22 b11 b12 b21 b22。

**輸出：** 結果矩陣兩行，每行兩個整數。

**範例輸入：**

```text
1 2 3 4
5 6 7 8
```

**範例輸出：**

```text
19 22
43 50
```

### C 作答

In [ ]:
%%c_answer Q84
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q84 C


In [ ]:
# 檢查 Q84 C：完成上方程式後執行
_run_check("Q84", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q84
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q84 C++


In [ ]:
# 檢查 Q84 C++：完成上方程式後執行
_run_check("Q84", "cpp")


## Q85. 二元一次方程

解方程 a1x+b1y=c1、a2x+b2y=c2。保證行列式不為 0。

**輸入：** 六個浮點數 a1 b1 c1 a2 b2 c2。

**輸出：** x 與 y，各到小數點後 3 位。

**範例輸入：**

```text
2 3 13 5 -2 4
```

**範例輸出：**

```text
2.000 3.000
```

### C 作答

In [ ]:
%%c_answer Q85
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q85 C


In [ ]:
# 檢查 Q85 C：完成上方程式後執行
_run_check("Q85", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q85
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q85 C++


In [ ]:
# 檢查 Q85 C++：完成上方程式後執行
_run_check("Q85", "cpp")


## Q86. 複數加減

讀入兩個複數 a+bi、c+di，輸出相加與相減後的實部、虛部。

**輸入：** 四個浮點數 a b c d。

**輸出：** 第一行為和，第二行為差；各到小數點後 2 位。

**範例輸入：**

```text
3.5 -2 7.25 4.5
```

**範例輸出：**

```text
10.75 2.50
-3.75 -6.50
```

### C 作答

In [ ]:
%%c_answer Q86
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q86 C


In [ ]:
# 檢查 Q86 C：完成上方程式後執行
_run_check("Q86", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q86
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q86 C++


In [ ]:
# 檢查 Q86 C++：完成上方程式後執行
_run_check("Q86", "cpp")


## Q87. 複數相乘

讀入兩個複數 a+bi、c+di，計算乘積。

**輸入：** 四個浮點數 a b c d。

**輸出：** 乘積的實部與虛部，各到小數點後 2 位。

**範例輸入：**

```text
3 -4 2.5 1.5
```

**範例輸出：**

```text
13.50 -5.50
```

### C 作答

In [ ]:
%%c_answer Q87
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q87 C


In [ ]:
# 檢查 Q87 C：完成上方程式後執行
_run_check("Q87", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q87
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q87 C++


In [ ]:
# 檢查 Q87 C++：完成上方程式後執行
_run_check("Q87", "cpp")


## Q88. 帶分數分解

讀入正分子 numerator 與正分母 denominator，輸出整數部分、餘數與原分母。

**輸入：** 兩個正整數。

**輸出：** whole remainder denominator。

**範例輸入：**

```text
157 24
```

**範例輸出：**

```text
6 13 24
```

### C 作答

In [ ]:
%%c_answer Q88
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q88 C


In [ ]:
# 檢查 Q88 C：完成上方程式後執行
_run_check("Q88", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q88
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q88 C++


In [ ]:
# 檢查 Q88 C++：完成上方程式後執行
_run_check("Q88", "cpp")


## Q89. 分數相加（不約分）

讀入 a/b 與 c/d，計算相加後的分子與分母，不必約分。

**輸入：** 四個非零分母的整數 a b c d。

**輸出：** 結果分子與分母。

**範例輸入：**

```text
5 12 7 18
```

**範例輸出：**

```text
174 216
```

### C 作答

In [ ]:
%%c_answer Q89
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q89 C


In [ ]:
# 檢查 Q89 C：完成上方程式後執行
_run_check("Q89", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q89
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q89 C++


In [ ]:
# 檢查 Q89 C++：完成上方程式後執行
_run_check("Q89", "cpp")


## Q90. 分數連乘（不約分）

讀入三個分數 a/b、c/d、e/f，計算乘積，不必約分。

**輸入：** 六個整數，分母不為 0。

**輸出：** 結果分子與分母。

**範例輸入：**

```text
2 3 5 7 9 11
```

**範例輸出：**

```text
90 231
```

### C 作答

In [ ]:
%%c_answer Q90
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q90 C


In [ ]:
# 檢查 Q90 C：完成上方程式後執行
_run_check("Q90", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q90
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q90 C++


In [ ]:
# 檢查 Q90 C++：完成上方程式後執行
_run_check("Q90", "cpp")


---

# Part G：綜合挑戰

## Q91. 灰階與亮度差

讀入 R、G、B（0 到 255），使用 gray=0.299R+0.587G+0.114B，四捨五入為整數；再輸出 gray−128。

**輸入：** 三個整數。

**輸出：** 灰階整數與相對 128 的差。

**範例輸入：**

```text
30 180 240
```

**範例輸出：**

```text
142 14
```

### C 作答

In [ ]:
%%c_answer Q91
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q91 C


In [ ]:
# 檢查 Q91 C：完成上方程式後執行
_run_check("Q91", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q91
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q91 C++


In [ ]:
# 檢查 Q91 C++：完成上方程式後執行
_run_check("Q91", "cpp")


## Q92. 六位數加權檢查碼

讀入六位正整數 abcdef，計算 (1a+3b+1c+3d+1e+3f)%10。

**輸入：** 一個六位正整數。

**輸出：** 檢查碼。

**範例輸入：**

```text
482731
```

**範例輸出：**

```text
7
```

### C 作答

In [ ]:
%%c_answer Q92
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q92 C


In [ ]:
# 檢查 Q92 C：完成上方程式後執行
_run_check("Q92", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q92
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q92 C++


In [ ]:
# 檢查 Q92 C++：完成上方程式後執行
_run_check("Q92", "cpp")


## Q93. ISBN-10 加權餘數

讀入九位正整數 d1...d9，計算 10d1+9d2+...+2d9 對 11 的餘數。

**輸入：** 一個九位正整數。

**輸出：** 加權總和對 11 的餘數。

**範例輸入：**

```text
123456789
```

**範例輸出：**

```text
1
```

### C 作答

In [ ]:
%%c_answer Q93
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q93 C


In [ ]:
# 檢查 Q93 C：完成上方程式後執行
_run_check("Q93", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q93
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q93 C++


In [ ]:
# 檢查 Q93 C++：完成上方程式後執行
_run_check("Q93", "cpp")


## Q94. 同日經過時間

讀入開始 h1:m1:s1 與結束 h2:m2:s2，保證結束不早於開始，計算經過時間。

**輸入：** 六個整數。

**輸出：** 經過的小時、分鐘、秒。

**範例輸入：**

```text
8 47 35 16 12 9
```

**範例輸出：**

```text
7 24 34
```

### C 作答

In [ ]:
%%c_answer Q94
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q94 C


In [ ]:
# 檢查 Q94 C：完成上方程式後執行
_run_check("Q94", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q94
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q94 C++


In [ ]:
# 檢查 Q94 C++：完成上方程式後執行
_run_check("Q94", "cpp")


## Q95. 跨日未來秒數

讀入目前 h:m:s 與經過秒數 passed，計算 24 小時制的新時刻。

**輸入：** 四個非負整數，時分秒合法。

**輸出：** 新時刻 h m s。

**範例輸入：**

```text
22 58 47 10000
```

**範例輸出：**

```text
1 45 27
```

### C 作答

In [ ]:
%%c_answer Q95
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q95 C


In [ ]:
# 檢查 Q95 C：完成上方程式後執行
_run_check("Q95", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q95
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q95 C++


In [ ]:
# 檢查 Q95 C++：完成上方程式後執行
_run_check("Q95", "cpp")


## Q96. 座標平移、縮放與旋轉

讀入點 (x,y)、平移量 (dx,dy) 與縮放係數 k。先平移、再以原點為中心縮放、最後逆時針旋轉 90°。

**輸入：** 五個浮點數 x y dx dy k。

**輸出：** 最終 x、y，各到小數點後 2 位。

**範例輸入：**

```text
3 -2 5 4 1.5
```

**範例輸出：**

```text
-3.00 12.00
```

### C 作答

In [ ]:
%%c_answer Q96
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q96 C


In [ ]:
# 檢查 Q96 C：完成上方程式後執行
_run_check("Q96", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q96
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q96 C++


In [ ]:
# 檢查 Q96 C++：完成上方程式後執行
_run_check("Q96", "cpp")


## Q97. 攝影畫面等比例縮放

讀入原寬 w、原高 h 與縮放比例 percent，計算新寬高與新像素總數，寬高四捨五入為整數。

**輸入：** 兩個正整數 w、h 與正浮點數 percent。

**輸出：** 新寬、新高、新像素數。

**範例輸入：**

```text
1920 1080 62.5
```

**範例輸出：**

```text
1200 675 810000
```

### C 作答

In [ ]:
%%c_answer Q97
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q97 C


In [ ]:
# 檢查 Q97 C：完成上方程式後執行
_run_check("Q97", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q97
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q97 C++


In [ ]:
# 檢查 Q97 C++：完成上方程式後執行
_run_check("Q97", "cpp")


## Q98. 資料傳輸完成時間

讀入檔案大小 MiB、傳輸速度 Mbps 與固定延遲 ms。1 byte=8 bits。計算總秒數。

**輸入：** 三個正浮點數。

**輸出：** 總時間到小數點後 3 位。

**範例輸入：**

```text
850 120 37
```

**範例輸出：**

```text
59.456
```

### C 作答

In [ ]:
%%c_answer Q98
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q98 C


In [ ]:
# 檢查 Q98 C：完成上方程式後執行
_run_check("Q98", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q98
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q98 C++


In [ ]:
# 檢查 Q98 C++：完成上方程式後執行
_run_check("Q98", "cpp")


## Q99. 三段行程平均速度

讀入三段距離 d1,d2,d3 與三段時間 t1,t2,t3，計算總平均速度與各段速度的算術平均。

**輸入：** 六個正浮點數。

**輸出：** 總平均速度與三段速度算術平均，各到小數點後 3 位。

**範例輸入：**

```text
120 80 150 2 1.5 3
```

**範例輸出：**

```text
53.846 54.444
```

### C 作答

In [ ]:
%%c_answer Q99
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q99 C


In [ ]:
# 檢查 Q99 C：完成上方程式後執行
_run_check("Q99", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q99
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q99 C++


In [ ]:
# 檢查 Q99 C++：完成上方程式後執行
_run_check("Q99", "cpp")


## Q100. 綜合訂單結算

讀入單價、數量、折扣%、運費、稅率%、分攤人數。先算商品小計並折扣，再加運費，最後加稅並平均分攤。

**輸入：** 單價、折扣、運費、稅率為浮點數；數量與人數為正整數。

**輸出：** 商品小計、折扣後、稅後總額、每人金額，各到小數點後 2 位。

**範例輸入：**

```text
129.9 18 12.5 80 5 4
```

**範例輸出：**

```text
2338.20 2045.93 2232.22 558.06
```

### C 作答

In [ ]:
%%c_answer Q100
#include <stdio.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行檢查：Q100 C


In [ ]:
# 檢查 Q100 C：完成上方程式後執行
_run_check("Q100", "c")


### C++ 作答

In [ ]:
%%cpp_answer Q100
#include <iostream>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行檢查：Q100 C++


In [ ]:
# 檢查 Q100 C++：完成上方程式後執行
_run_check("Q100", "cpp")


---

# 總檢查

In [ ]:
_final_report()
